# Rondas Hídricas — Acotamiento y Sistema de Información del Recurso Hídrico (SIRH)

> **Contexto de dominio:** [`docs/fuentes/rondas_hidricas.md`](../../docs/fuentes/rondas_hidricas.md)
> **Bloque:** A — Gestión | **Línea:** `rondas_hidricas`
> **Variables principales:** caudal (m³/s), nivel del agua (m), ancho de faja paralela (m)
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST
> **Flujo:** `Plan.md` sección 3 — ciclo estadístico completo.

---

## ¿Qué son las rondas hídricas en Colombia?

Las **rondas hídricas** son fajas de terreno de transición (ecotonos) entre los ecosistemas terrestres y acuáticos. Cumplen funciones ecosistémicas esenciales: regulan los caudales de creciente, conservan la biodiversidad riparia, filtran sedimentos y contaminantes, y reducen el riesgo de inundación. Históricamente se definían como una franja estática de 30 metros a cada lado del cauce; sin embargo, la **Resolución 957 de 2018 (MADS)** reemplazó este criterio por un **acotamiento funcional y multidimensional**, basado en criterios geomorfológicos, hidrológicos y ecosistémicos.

El **SIRH** (Sistema de Información del Recurso Hídrico) es el repositorio oficial donde las CARs reportan los avances de acotamiento, junto con datos de concesiones, vertimientos y calidad del agua. El análisis estadístico del caudal es fundamental para dos propósitos: (1) delimitar el área de inundación para crecientes de período de retorno 100 años, y (2) monitorear el estado hídrico de las fuentes dentro de la ronda.

## Marco normativo clave

| Norma | Contenido relevante |
|---|---|
| **Ley 1450 de 2011, Art. 206** | Asigna a las CARs la competencia para acotar rondas hídricas |
| **Decreto 2245 de 2017** | Define criterios técnicos funcionales para el acotamiento |
| **Resolución 957 de 2018 (MADS)** | Adopta la guía técnica de acotamiento; reemplaza el criterio fijo de 30 m |
| **Decreto 303 de 2012** | Reglamenta el Registro de Usuarios del Recurso Hídrico (RURH) |

## Instituciones responsables

| Institución | Rol |
|---|---|
| **MADS** | Rector de política; emite la guía técnica de acotamiento (Res. 957/2018) |
| **IDEAM** | Administra el SIRH; provee información hidroclimatológica oficial |
| **CARs** | Ejecutan los estudios de acotamiento y reportan al SIRH |
| **Municipios** | Incorporan el acotamiento en sus POT como determinante ambiental |

## Preguntas que responde este análisis

1. ¿Cuál es el régimen de caudales en el cauce de interés? ¿Cuáles son los caudales de creciente ordinaria y extraordinaria?
2. ¿Existe una tendencia de cambio en el caudal medio mensual o anual (Mann-Kendall)?
3. ¿Qué tan variable es el caudal interanualmente? ¿Se puede atribuir parte de esa variabilidad al ENSO?
4. ¿Qué modelo estadístico proyecta mejor el caudal para calibrar modelos hidráulicos (HEC-RAS)?
5. ¿Cuáles son los meses de mayor riesgo de creciente que deben considerarse para el manejo de la ronda?

---

**Antes de ejecutar:** Leer `docs/fuentes/rondas_hidricas.md` para entender los indicadores QBR y RFV de calidad de bosque ripario, los métodos de análisis de frecuencia de caudales máximos y las fuentes de datos requeridas.

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "rondas_hidricas"
VARIABLE = "caudal"
UNIDAD = "m³/s"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio

Esta celda carga la ficha técnica de la línea `rondas_hidricas` directamente desde `docs/fuentes/`. La ficha contiene el resumen del SIRH y del proceso de acotamiento funcional de rondas hídricas, las variables relevantes, los indicadores de bosque ripario y la normativa aplicable.

Se recomienda revisar especialmente:
- La tabla de **Variables ambientales**: caudal (m³/s o L/s), nivel del agua (m), precipitación (mm), ancho de faja paralela (m), altura del dosel (m).
- Los **indicadores**: QBR (calidad del bosque de ribera) y RFV (estado del ecosistema ribereño).
- Las **fuentes de datos**: MDE, LiDAR, series hidroclimatológicas del IDEAM (mínimo 15 años), datos administrativos del RURH en el SIRH.

> **Nota sobre outliers (ADR-002):** Los picos de caudal durante eventos de creciente son la señal más importante en esta línea: definen el ancho de la zona de flujo preferente y la extensión de la ronda hídrica. Eliminar estos valores máximos introduciría errores graves en el análisis de frecuencia de caudales máximos (distribuciones Gumbel, Log-Pearson III).

> **Nota sobre series cortas:** La Res. 957 de 2018 requiere series mínimas de 15 años. Si los datos disponibles son menores, documentar la limitación y aplicar regionalización estadística a partir de cuencas vecinas con mayor cobertura instrumental.

In [ ]:
ficha = DOCS_FUENTES / "rondas_hidricas.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### ¿Qué datos necesito?

Para el análisis de rondas hídricas se requiere una serie temporal de caudales del cauce a acotar:

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | datetime | Fecha de la medición (diaria o mensual) |
| `caudal` | float | Caudal (m³/s) — incluir máximos históricos |
| `nivel` | float (opcional) | Nivel del agua en el cauce (m) |
| `precipitacion` | float (opcional) | Precipitación acumulada (mm) |
| `estacion` | str (opcional) | Código IDEAM de la estación hidrológica cercana |

### Fuentes oficiales de datos en Colombia

- **IDEAM — DHIME:** [dhime.ideam.gov.co](http://dhime.ideam.gov.co) — Series de caudales diarios y niveles. Se requiere mínimo 15 años de registro continuo (Res. 957/2018).
- **IDEAM — SIRH:** [sirh.ideam.gov.co](http://sirh.ideam.gov.co) — Registro de Usuarios del Recurso Hídrico (RURH).
- **CARs regionales:** Limnígrafos y datos batimétricos propios. Ej. CAR Cundinamarca, CORNARE, CORPOAMAZONIA.
- **Fotografías aéreas históricas / SAR:** Para reconstrucción de huellas de inundación históricas donde no hay registros instrumentales.

### Requerimientos mínimos para el acotamiento (Res. 957/2018)

La Resolución 957 de 2018 establece que el análisis hidrológico para el acotamiento debe incluir:

1. **Análisis de frecuencia de caudales máximos** con distribuciones de probabilidad extremas (Gumbel, Log-Pearson III).
2. **Estimación del caudal de creciente ordinaria** (período de retorno Tr = 1 a 2 años): define el límite del cauce permanente.
3. **Estimación del caudal de creciente extraordinaria** (Tr = 100 años): define la extensión máxima de la ronda.
4. **Modelación hidráulica** (HEC-RAS 1D/2D) para traducir el caudal estimado en mancha de inundación.

### Frecuencia y horizonte recomendados

- **Frecuencia:** Diaria para análisis de caudales máximos. Mensual para análisis de tendencias.
- **Horizonte mínimo reglamentario:** 15 años (Res. 957/2018). Recomendado: 30+ años.

> **Nota:** El dataset sintético de ejemplo usa distribución Gamma. Para el análisis real de rondas, es fundamental tener acceso a los caudales máximos instantáneos (no solo promedios), ya que son los que determinan la mancha de inundación.

In [ ]:
# df = load_csv("data/raw/rondas_hidricas.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "caudal": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

### ¿Qué hace `validate()` en esta línea?

La función `validate()` aplica rangos físicos específicos para la línea `rondas_hidricas` definidos en `config.py`. Verifica que:

- El caudal no sea negativo.
- No haya fechas duplicadas ni saltos temporales anómalos.
- Los niveles del agua estén dentro de rangos físicos del cauce.
- La serie tenga suficiente longitud (la Res. 957/2018 exige al menos 15 años de registro).

### Señales de alerta en la validación

Si `val.summary()` reporta problemas:

- **Series menores a 15 años:** No cumplen el requisito mínimo de la Res. 957/2018. Documentar la limitación y evaluar regionalización estadística.
- **Caudales cero:** En ríos de régimen permanente es señal de fallo de sensor o error de datos. En ríos intermitentes o de régimen semipermanente, puede ser real.
- **Picos de caudal extremos:** No son errores. Son los eventos de creciente que determinan el ancho de la ronda hídrica.

> **ADR-006:** El parámetro `linea_tematica="rondas_hidricas"` activa validaciones enfocadas en la calidad de series para análisis de frecuencia de caudales máximos.

> **ADR-002:** Los caudales máximos son LA señal de interés en el estudio de rondas hídricas. Bajo ninguna circunstancia aplicar clipping o eliminación de valores extremos sin justificación técnica documentada explícitamente en `docs/decisiones.md`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_rondas_hidricas.html",
       title="EDA — Rondas Hídricas", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### ¿Qué buscar en las gráficas?

Las visualizaciones de esta sección permiten caracterizar el régimen hidrológico del cauce de interés, paso fundamental antes del análisis de frecuencia de caudales máximos para el acotamiento de la ronda hídrica. Se deben identificar:

**En la serie temporal (plot_series):**
- Magnitud y frecuencia de crecientes: ¿Con qué frecuencia se registran caudales que superan el umbral de creciente ordinaria? ¿Hay eventos que claramente superan la capacidad del cauce permanente?
- Tendencia: ¿Los caudales máximos han aumentado en las últimas décadas? Esto puede indicar pérdida de cobertura boscosa en la cuenca, urbanización o cambios en el régimen de lluvia por cambio climático.
- Eventos extremos identificables: picos asociados a La Niña suelen ser los más relevantes para el acotamiento porque definen la mancha de inundación de mayor extensión.

**En las medias estacionales (plot_seasonal_means):**
- Identificar los meses de mayor riesgo de creciente: en la Región Andina suelen ser marzo-abril y octubre-noviembre.
- Estacionalidad del caudal mínimo: definir el caudal de estiaje que permite establecer el límite inferior del cauce permanente (banca llena).
- Variabilidad mensual: alta variabilidad (cociente Q máx / Q mín > 10) indica cauce con dinámica hidráulica compleja que requiere análisis detallado para el acotamiento.

In [ ]:
plot_series(df, "fecha", "caudal", title="Rondas Hídricas — caudal (m³/s)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "caudal", period="month")
plt.show()

## 3c. HAND y analisis de frecuencia — Delimitacion de ronda hidrica

El **HAND (Height Above Nearest Drainage)** es un indice geomorfometrico derivado del DEM que expresa la diferencia de elevacion entre cada celda y el drenaje mas cercano. Es la herramienta central para el acotamiento funcional de rondas hidricas (Res. 957/2018):

```
HAND(x,y) = elevacion(x,y) - elevacion(drenaje_mas_cercano)
HAND = 0    : cauce permanente
HAND < 1m   : zona inundacion banca llena (ordinaria)
HAND < Tr100: zona de flujo preferente (inundacion 100 anos)
```

Herramientas: **WhiteboxTools** (algoritmo D8 para red drenaje + HAND) o TauDEM.

**Analisis de frecuencia de caudales maximos (Gumbel):**
```
F(Q) = exp(-exp(-y))    y = (Q - alpha) / beta
Caudal_Tr = alpha - beta * ln(-ln(1 - 1/Tr))
```

In [ ]:
# HAND conceptual + analisis frecuencia Gumbel para caudales maximos
from scipy import stats
np.random.seed(9)

# -- Simulacion HAND transecto de rio (perfil transversal) ----------------
distancia_m = np.linspace(-200, 200, 400)   # m desde el cauce
# Perfil topografico sintetico (valle en V con llanura)
hand_perfil = np.where(
    np.abs(distancia_m) < 30,
    np.abs(distancia_m) * 0.05,             # cauce y zona banca llena
    0.05 * 30 + (np.abs(distancia_m) - 30) * 0.04  # ladera
) + np.random.normal(0, 0.1, 400)
hand_perfil = np.clip(hand_perfil, 0, None)

# Umbrales HAND para ronda hidrica
HAND_BANCA_LLENA = 0.5  # m — inundacion ordinaria
HAND_TR15 = 1.8         # m — periodo retorno 15 anos
HAND_TR100 = 4.2        # m — periodo retorno 100 anos (zona flujo preferente)

# -- Analisis frecuencia Gumbel sobre caudales maximos anuales ------------
n_anos = 35
q_max_anual = np.random.gumbel(loc=80, scale=25, size=n_anos).clip(20)  # m3/s

# Ajuste Gumbel (metodo momentos)
alpha_g = q_max_anual.std() * np.sqrt(6) / np.pi
beta_g  = q_max_anual.mean() - 0.5772 * alpha_g

Tr_periodos = [2, 5, 10, 25, 50, 100]
q_tr = [beta_g - alpha_g * np.log(-np.log(1 - 1/tr)) for tr in Tr_periodos]
df_gumbel = pd.DataFrame({'Tr_anos': Tr_periodos, 'Q_m3s': [round(q,2) for q in q_tr]})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Perfil HAND transecto
ax = axes[0]
ax.fill_between(distancia_m, hand_perfil, alpha=0.3, color='#8B4513')
ax.plot(distancia_m, hand_perfil, color='#8B4513', lw=1)
ax.axhline(HAND_BANCA_LLENA, color='#3498db', ls='-', lw=2, label=f'Banca llena (HAND={HAND_BANCA_LLENA}m)')
ax.axhline(HAND_TR15, color='orange', ls='--', lw=1.5, label=f'Tr=15 anos (HAND={HAND_TR15}m)')
ax.axhline(HAND_TR100, color='red', ls='--', lw=1.5, label=f'Tr=100 anos/ronda hidrica (HAND={HAND_TR100}m)')
ax.set_xlabel('Distancia al cauce (m)'); ax.set_ylabel('HAND (m)')
ax.set_title('HAND — Perfil transversal ronda hidrica\n(WhiteboxTools D8, Res. 957/2018)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel B: Curva de frecuencia Gumbel
ax = axes[1]
Tr_cont = np.linspace(1.1, 200, 500)
q_cont = beta_g - alpha_g * np.log(-np.log(1 - 1/Tr_cont))
ax.plot(Tr_cont, q_cont, lw=2, color='#2980b9', label='Ajuste Gumbel')
ax.scatter([r['Tr_anos'] for _, r in df_gumbel.iterrows()],
           [r['Q_m3s'] for _, r in df_gumbel.iterrows()],
           color='red', zorder=5, s=60, label='Tr de diseno')
ax.axvline(100, color='red', ls='--', lw=1.5, label='Tr=100 anos (ronda hidrica)')
ax.axvline(15, color='orange', ls='--', lw=1, label='Tr=15 anos (Res. 957/2018)')
ax.set_xscale('log')
ax.set_xlabel('Periodo de retorno Tr (anos, log)'); ax.set_ylabel('Caudal maximo (m3/s)')
ax.set_title('Analisis frecuencia Gumbel — Caudales maximos\nRonda hidrica: zona de flujo preferente Tr=100', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print('Caudales de diseno por periodo de retorno:')
print(df_gumbel.to_string(index=False))

## 3b. Covariable ENSO (ONI) — Por qué importa en el acotamiento de rondas

### ¿Por qué el ENSO es relevante para las rondas hídricas?

Las rondas hídricas se delimitan en función de la creciente extraordinaria (Tr = 100 años). La variabilidad climática interanual asociada al ENSO modifica la frecuencia y magnitud de los eventos de creciente, afectando directamente los análisis de frecuencia de caudales máximos:

- **La Niña** genera eventos de precipitación extrema con mayor frecuencia. En años de La Niña fuerte (ONI < -1.5), el caudal máximo mensual puede superar varias veces el promedio histórico. Esto eleva los cuantiles de frecuencia y puede ampliar la mancha de inundación estimada para Tr = 100 años.
- **El Niño** reduce los caudales y disminuye el riesgo de inundación fluvial, pero puede incrementar los incendios forestales en las zonas de ribera y deteriorar la cobertura vegetal de la ronda.

El análisis ENSO con `enso_lagged()` permite identificar si los eventos de creciente registrados en la serie histórica coinciden con fases de La Niña, y si hay una señal de intensificación que deba considerarse en los escenarios de cambio climático del acotamiento.

### Lag para rondas hídricas (ADR-007)

Para cuerpos de agua superficiales, la respuesta del caudal al ENSO es más rápida que en acuíferos. El lag configurado en `config.ENSO_LAG_MESES["rondas_hidricas"]` refleja el tiempo de respuesta hidrológica de cuencas superficiales. La señal ENSO llega al caudal con menos desfase que a los niveles piezométricos subterráneos.

> **ADR-007:** `enso_lagged()` aplica el lag correcto y agrega las columnas `oni_lag` y `enso_fase` al DataFrame. El uso de `enso_dummy()` (ONI sin lag) sobreestima la señal predictiva inmediata y no captura correctamente la dinámica hidrológica real.

### Uso en el análisis de rondas

Las columnas ENSO generadas permiten:
1. Verificar si los años con mayores caudales máximos fueron años de La Niña — validación empírica del análisis de frecuencia.
2. Estratificar el análisis de caudales máximos por fase ENSO para evaluar la sensibilidad del acotamiento a la variabilidad climática.
3. Argumentar técnicamente ante las CARs si la ronda hídrica debe ser ampliada en escenarios de La Niña futura intensificada por cambio climático.

In [ ]:
# --- Covariable ENSO (lag específico para Rondas Hídricas) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="rondas_hidricas")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial — Estacionariedad y tendencia para rondas hídricas

### ¿Por qué verificar estacionariedad? (ADR-004)

Para el análisis de rondas hídricas, el interés principal no está en el caudal medio sino en los **caudales máximos**. Sin embargo, la verificación de estacionariedad en la serie de caudal mensual es útil para:

1. Detectar si hay una tendencia en los caudales que afecte la estimación de cuantiles de frecuencia.
2. Verificar si los modelos de series temporales para proyección de caudal son estadísticamente válidos.
3. Identificar posibles no estacionariedades asociadas a cambios en el uso del suelo de la cuenca.

| Prueba | H₀ | Interpretación para rondas |
|---|---|---|
| **ADF** | Serie no estacionaria | Si rechaza: caudal es estacionario; los cuantiles son estables en el tiempo |
| **KPSS** | Serie es estacionaria | Si rechaza: hay tendencia; los cuantiles históricos pueden no ser representativos del futuro |

### Implicaciones de la tendencia para el acotamiento (Mann-Kendall)

- `trend = "increasing"` con p < 0.05 en caudal: los caudales máximos han aumentado con el tiempo — posiblemente por deforestación de la cuenca o impermeabilización urbana. **La mancha de inundación para Tr = 100 años debe recalcularse con los datos más recientes**, dando más peso a los años recientes.
- `trend = "decreasing"`: los caudales máximos han disminuido — posible efecto de reforestación, o que las crecientes más severas ocurrieron en el inicio de la serie.
- `slope` positivo significativo: la ronda hídrica debería ser más amplia de lo que sugiere el análisis con toda la serie histórica sin ponderar temporalmente.

> **ADR-004:** Obligatorio ADF + KPSS antes de proyectar caudales con ARIMA. Para el análisis de frecuencia de caudales máximos (componente hidráulica del acotamiento), usar distribuciones de valores extremos (Gumbel, GEV, Log-Pearson III) que tienen sus propios supuestos estadísticos distintos de los tests de estacionariedad.

In [ ]:
ts = df.set_index("fecha")["caudal"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias — Umbrales de creciente y caudal ambiental

### ¿Qué umbrales son relevantes para rondas hídricas?

A diferencia de la calidad del agua, en rondas hídricas los umbrales críticos son **umbrales de flujo hidráulico** que definen el comportamiento del cauce y la extensión de la ronda:

| Umbral de caudal | Definición | Importancia para el acotamiento |
|---|---|---|
| **Caudal mínimo ambiental** | ~25% del caudal medio anual (metodología Tennant) | Define el límite inferior del cauce permanente (banca llena mínima) |
| **Caudal de creciente ordinaria** (Tr 1-2 años) | Cuantil 90-95% de la distribución anual de máximos | Límite del cauce permanente en el acotamiento |
| **Caudal de diseño** (Tr 25-50 años) | Cuantil de distribución de valores extremos | Dimensionamiento de obras de protección menores |
| **Caudal de creciente extraordinaria** (Tr 100 años) | Cuantil extremo de distribución Gumbel/Log-Pearson | Extensión máxima de la ronda hídrica (Decreto 2245/2017) |

### Estimación empírica de cuantiles de caudal

Con la serie disponible, una aproximación a los cuantiles de creciente:
```python
# Cuantiles de la distribución de caudales
q_ordinaria = df["caudal"].quantile(0.90)   # Creciente ordinaria ~ Tr 1-2 años
q_extraordinaria = df["caudal"].quantile(0.99)  # Aproximación extrema

print(f"Caudal ordinaria (~Tr 2 años):      {q_ordinaria:.2f} m³/s")
print(f"Caudal extraordinaria (~Tr 100 años): {q_extraordinaria:.2f} m³/s")
# NOTA: Para el acotamiento real, usar distribuciones Gumbel o Log-Pearson III
# ajustadas sobre la serie de máximos anuales — no solo percentiles empíricos.
```

### El caudal ambiental como límite inferior

El caudal ambiental no es el caudal mínimo registrado, sino el caudal mínimo **necesario** para mantener la funcionalidad ecológica del cauce. La Resolución 957/2018 no define un método único, pero el IDEAM recomienda la metodología Tennant modificada para Colombia: entre el 10% y el 30% del caudal medio anual según la zona de vida.

> **ADR-005:** Los umbrales de IUA para caudal ambiental están en `config.IUA_THRESHOLDS`. Para la metodología de caudal ambiental específica por cuenca, documentar la decisión metodológica en `docs/decisiones.md`.

In [ ]:
rep = exceedance_report(df["caudal"], variable="caudal")
if rep.empty:
    print("Sin normas colombianas registradas para 'caudal'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["caudal"], method="linear")
print(f"Faltantes antes: {df["caudal"].isna().sum()} | "
      f"después: {df_clean["caudal"].isna().sum()}")

## 7. Modelos predictivos — Métricas para análisis de caudales en rondas

### Contexto del modelado en rondas hídricas

En el contexto del acotamiento, los modelos de series temporales tienen un propósito diferente al de la hidrología de gestión: no se busca predecir el caudal medio futuro, sino **entender el régimen histórico** y **proyectar el caudal en escenarios hipotéticos de cambio climático**. El foco está en los extremos, no en la media.

### NSE y KGE como métricas primarias

| Métrica | Importancia para rondas |
|---|---|
| **NSE** | Verifica que el modelo replica correctamente los períodos secos y húmedos |
| **KGE** | Evalúa si el modelo captura la variabilidad del caudal (α) — crucial para extremos |
| **MAE** | Error medio absoluto — complementa NSE en series con eventos extremos |
| **RMSE** | Referencia secundaria; sensible a los picos que son justamente los más importantes |

### Por qué el RMSE no es suficiente

Un modelo que prediga bien los caudales medios pero subestime los picos tendrá RMSE aceptable, pero **subestimará el caudal de creciente para Tr = 100 años**, resultando en una ronda hídrica más estrecha de lo necesario. Esto expone infraestructura y comunidades a inundaciones que la norma debería haber protegido.

El componente **α del KGE** (relación de variabilidades) es el indicador más crítico: si α < 0.7, el modelo subestima sistemáticamente los picos, lo que invalida su uso para inferir caudales extremos.

### ¿Qué modelo es más adecuado para rondas?

- **SARIMA:** Útil para entender la estructura temporal y detectar tendencias en el caudal medio. No es el mejor para reproducir picos.
- **Prophet:** Robusto ante series con brechas — común en cuencas sin estaciones continuas. Captura tendencias y estacionalidad anual.
- **XGBoost con rezagos:** Puede capturar la respuesta no lineal del caudal a la precipitación antecedente.

> **ADR-003:** Para caudal (variable positiva), NSE y KGE son las métricas primarias. No usar RMSLE. Para el análisis de frecuencia de caudales máximos propiamente dicho (Gumbel, Log-Pearson III), usar software especializado (R: fitdistrplus, Python: scipy.stats) además de este notebook.

In [ ]:
ts = df_clean.set_index("fecha")["caudal"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones metodológicas

### Síntesis del ciclo estadístico para rondas hídricas

Este notebook implementó el ciclo estadístico completo para la línea `rondas_hidricas`:

1. **Validación de dominio** (`validate()`): verificación de series con longitud mínima reglamentaria.
2. **EDA**: caracterización del régimen de caudales y frecuencia de crecientes.
3. **ENSO con lag** (`enso_lagged()`): análisis de la influencia climática sobre los eventos extremos.
4. **Estacionariedad** (ADF + KPSS): verificación pre-ARIMA (ADR-004).
5. **Tendencia** (Mann-Kendall): detección de tendencias en caudales máximos para el acotamiento.
6. **Excedencias**: umbrales de creciente ordinaria (Tr 2 años) y extraordinaria (Tr 100 años).
7. **Modelos** (SARIMA, Prophet, XGBoost): proyecciones con métricas NSE, KGE (α crítico).

### Decisiones metodológicas clave

| Decisión | Regla aplicada | ADR |
|---|---|---|
| Conservar picos de caudal máximo | Son LA señal para el acotamiento; no son outliers | ADR-002 |
| ADF + KPSS obligatorios | Verificar estacionariedad antes de modelar | ADR-004 |
| Series mínimo 15 años | Requisito Res. 957/2018 | ADR-006 |
| ENSO con lag | Verificar si extremos coinciden con La Niña | ADR-007 |
| NSE y KGE primarios; α del KGE crítico | α < 0.7 invalida el modelo para extremos | ADR-003 |

### Hallazgos de referencia (datos sintéticos)

Al trabajar con datos reales, documentar:
- Longitud de la serie (¿cumple el mínimo de 15 años?).
- Tendencia Mann-Kendall en caudales máximos mensuales.
- Percentiles Q90 y Q99 como proxy de creciente ordinaria y extraordinaria.
- Coincidencia de eventos extremos con fases ENSO (La Niña).
- Mejor modelo por NSE y KGE (especialmente α).
- Registrar decisiones en `docs/decisiones.md`.

### Normativa y referencias
- `docs/fuentes/rondas_hidricas.md` — Ficha técnica completa
- Ley 1450 de 2011, Art. 206 — Competencia CARs para acotamiento
- Decreto 2245 de 2017 — Criterios técnicos funcionales de acotamiento
- Resolución 957 de 2018 (MADS) — Guía técnica de acotamiento de rondas hídricas
- IDEAM — SIRH / RURH — Fuentes de datos y reportes institucionales

## 9. Cómo adaptar a datos reales

Esta sección es la guía práctica para técnicos de CARs o consultoras que estén ejecutando un estudio de acotamiento de rondas hídricas bajo la Resolución 957 de 2018.

### Paso 1 — Obtener los datos

**Fuentes recomendadas para el análisis hidrológico del acotamiento:**

| Tipo de dato | Fuente | Cómo acceder |
|---|---|---|
| Caudales diarios (mínimo 15 años) | IDEAM — DHIME | dhime.ideam.gov.co |
| Niveles del agua | IDEAM — DHIME (limnígrafos) | dhime.ideam.gov.co |
| Caudales máximos instantáneos | IDEAM — Anuarios hidrológicos | Publicaciones IDEAM o DHIME |
| MDE / LiDAR | IGAC / Aerocivil | geoportal.igac.gov.co |
| Huellas de inundación históricas | CARs / UNGRD / Fotografías aéreas | Solicitud directa |
| Datos SAR (Sentinel-1) | Copernicus / ESA | scihub.copernicus.eu (gratuito) |

### Paso 2 — Preparar el archivo CSV de caudales

```
fecha,caudal,nivel,precipitacion,estacion
1990-01-01,5.2,1.2,95.0,32097010
1990-02-01,4.8,1.1,80.0,32097010
...
```

- `fecha`: formato ISO, primer día del mes para series mensuales.
- `caudal`: caudal medio mensual (m³/s).
- `nivel`: nivel del agua (m) — si está disponible.
- `estacion`: código IDEAM de la estación hidrológica del cauce a acotar.
- **Importante:** Para el análisis de frecuencia, es mejor extraer también los **caudales máximos instantáneos anuales** del DHIME.
- Guardar en `data/raw/rondas_hidricas.csv`.

### Paso 3 — Activar la carga real

```python
df = load_csv("data/raw/rondas_hidricas.csv", date_col="fecha")
```

### Paso 4 — Extraer serie de máximos anuales para análisis de frecuencia

El análisis de frecuencia requiere la serie de **máximos anuales**, no la serie completa:

```python
# Máximos anuales para análisis de frecuencia de crecientes
maximos_anuales = df.set_index("fecha")["caudal"].resample("YE").max().dropna()
print(f"Serie de máximos anuales: {len(maximos_anuales)} años")
print(f"Q máximo histórico: {maximos_anuales.max():.2f} m³/s")
print(f"Q percentil 90 (proxy Tr~10 años): {maximos_anuales.quantile(0.90):.2f} m³/s")
print(f"Q percentil 99 (proxy Tr~100 años): {maximos_anuales.quantile(0.99):.2f} m³/s")
# Para el acotamiento real, usar scipy.stats.gumbel_r o lmoments
# sobre esta serie de máximos anuales
```

### Paso 5 — Verificar cumplimiento del requisito de 15 años (Res. 957/2018)

```python
n_anios = (df["fecha"].max() - df["fecha"].min()).days / 365.25
if n_anios < 15:
    print(f"ADVERTENCIA: Serie de {n_anios:.1f} años — menor al mínimo de 15 años.")
    print("Considerar regionalización estadística con cuencas vecinas instrumentadas.")
else:
    print(f"OK: Serie de {n_anios:.1f} años — cumple Res. 957/2018.")
```

### Checklist de verificación antes de entregar a la CAR

- [ ] Serie validada con mínimo 15 años de registro (ADR-006 y Res. 957/2018).
- [ ] Caudales máximos extremos conservados (ADR-002).
- [ ] ADF + KPSS ejecutados en serie de caudal mensual (ADR-004).
- [ ] ENSO incorporado con `enso_lagged()` (ADR-007).
- [ ] Cuantiles de creciente ordinaria (Q90) y extraordinaria (Q99) calculados.
- [ ] NSE y KGE del modelo con α > 0.7 (ADR-003).
- [ ] Nota: el acotamiento final requiere modelación hidráulica HEC-RAS fuera de este notebook.
- [ ] Decisiones metodológicas registradas en `docs/decisiones.md`.